# Core Pipeline for A549 dataset (QC, Integration, Clustering)

Clean pipeline from raw inputs to integrated h5ad.


## 1. Environment and versions
Log key package versions for reproducibility.

In [ ]:
# Environment + determinism (set before importing heavy libs)
import os

# Best-effort determinism across machines (threads can introduce tiny non-determinism)
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import scanpy as sc
import harmonypy
import leidenalg
import igraph
import umap

from importlib.metadata import PackageNotFoundError, version


def _pkg_version(dist_name: str, module=None) -> str:
    try:
        return version(dist_name)
    except PackageNotFoundError:
        return getattr(module, "__version__", "unknown")


sc.settings.n_jobs = 1
sc.logging.print_header()
print(f"scanpy: {_pkg_version('scanpy', sc)}")
print(f"harmonypy: {_pkg_version('harmonypy', harmonypy)}")
print(f"leidenalg: {_pkg_version('leidenalg', leidenalg)}")
print(f"python-igraph: {_pkg_version('igraph', igraph)}")
print(f"umap-learn: {_pkg_version('umap-learn', umap)}")


## 2. Parameters
Define all thresholds and settings here.

In [ ]:
# QC thresholds
MIN_GENES = 200
MAX_GENES = 7000
MAX_MT_PCT = 10
MIN_CELLS_PER_GENE = 3

# Clustering/UMAP
N_PCS = 40
LEIDEN_RES = [0.3, 0.6, 0.9, 1.2]
NEIGHBORS_UMAP_SEED = 0  # matches original Scanpy default
LEIDEN_SEED = 42  # matches original clustering notebook (03_Notebooks/02_A549_Clustering.ipynb)

# Harmony
HARMONY_THETA = 2
HARMONY_MAX_ITER = 100

# Reproducibility
import random
import numpy as np
random.seed(NEIGHBORS_UMAP_SEED)
np.random.seed(NEIGHBORS_UMAP_SEED)

# Output control
OVERWRITE_OUTPUT = False  # set True to recompute + overwrite data/figure6/runtime/processed/A549_integrated_final.h5ad


## 3. Paths
Set input/output directories.

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
for p in [CWD, *CWD.parents]:
    if (p / "code" / "figure6" / "scripts" / "project_paths.py").exists():
        REPO_ROOT = p
        break
else:
    raise RuntimeError(
        "Could not locate the repository root. Launch Jupyter from within the repo (or a subdirectory) so code/figure6/notebooks is visible."
    )

if str(REPO_ROOT / "code" / "figure6") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "code" / "figure6"))

from scripts.project_paths import get_notebook_paths

PATHS = get_notebook_paths("01_Core_Pipeline", REPO_ROOT)
PROJECT_ROOT = PATHS.common.repo_root
DIR_CODE = PATHS.common.code_dir
DIR_DATA = PATHS.common.data_dir
DIR_DOCS = PATHS.common.docs_dir
DIR_FIGURES_ROOT = PATHS.common.figures_root
DIR_RUNTIME = PATHS.common.runtime_dir
DIR_DOWNLOADS = PATHS.common.downloads_dir
DIR_RAW = PATHS.common.raw_dir
DIR_PROC_OUT = PATHS.common.processed_dir
DIR_SAVE = PATHS.save_dir
DIR_FIG_SAVE = PATHS.figure_dir

sc.settings.verbosity = 3
sc.settings.figdir = str(DIR_FIG_SAVE)


## 3.1 Data acquisition (GEO)
This notebook downloads the GEO raw archive and prepares runtime inputs under data/figure6/runtime/.

- data/figure6/runtime/downloads/GSE161363.tar (cached)
- data/figure6/runtime/downloads/GSE161363_RAW/ (extracted)
- data/figure6/runtime/raw/M5k/ and data/figure6/runtime/raw/LM0/ (prepared for the loader)


In [ ]:
# Download + prepare raw inputs under data/figure6/runtime/
# This cell is safe to skip if DIR_RAW/M5k and DIR_RAW/LM0 already exist.

import gzip
import os
import tarfile
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

GSE_ACCESSION = 'GSE161363'
GEO_URL = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE161363&format=file'

ARCHIVE_PATH = DIR_DOWNLOADS / f'{GSE_ACCESSION}.tar'
EXTRACT_DIR = DIR_DOWNLOADS / f'{GSE_ACCESSION}_RAW'

# Expected prepared sample dirs (used by the loader)
M5K_DIR = DIR_RAW / 'M5k'
LM0_DIR = DIR_RAW / 'LM0'

# Helper: read MatrixMarket dimensions from a .mtx.gz

def read_mtx_gz_nrows(path: Path) -> int:
    with gzip.open(path, 'rt') as f:
        # skip comments
        for line in f:
            if not line.startswith('%'):
                nrows, ncols, nnz = map(int, line.strip().split())
                return nrows
    raise ValueError(f'Could not read matrix dimensions from {path}')


def gunzip_to(src_gz: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(src_gz, 'rb') as f_in, open(dst, 'wb') as f_out:
        f_out.write(f_in.read())


def ensure_prepared_sample(
    extract_dir: Path,
    sample_id: str,
    tag: str,
    out_dir: Path,
    genes_hint: Path | None = None,
) -> None:
    """Prepare <sample_id> into out_dir with *_barcodes.tsv/_genes.tsv/_matrix.mtx/_meta.tsv.

    For M5k, GEO may not provide a genes.5k file; we auto-select a genes file with matching n_genes.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    bc_gz = extract_dir / f'{sample_id}_barcodes.{tag}.tsv.gz'
    mx_gz = extract_dir / f'{sample_id}_matrix.{tag}.mtx.gz'
    meta_gz = extract_dir / f'{sample_id}_meta.{tag}.tsv.gz'

    if not bc_gz.exists() or not mx_gz.exists() or not meta_gz.exists():
        raise FileNotFoundError(f'Missing expected GEO files for {sample_id} tag={tag} under {extract_dir}')

    # genes file: prefer matching sample_id, else choose by matrix nrows
    genes_gz = extract_dir / f'{sample_id}_genes.{tag}.tsv.gz'
    if not genes_gz.exists():
        nrows = read_mtx_gz_nrows(mx_gz)
        candidates = sorted(extract_dir.glob('GSM*_genes.*.tsv.gz'))
        picked = None
        for cand in candidates:
            # count lines cheaply
            with gzip.open(cand, 'rt') as f:
                n = sum(1 for _ in f)
            if n == nrows:
                picked = cand
                break
        if picked is None and genes_hint and genes_hint.exists():
            picked = genes_hint
        if picked is None:
            raise FileNotFoundError(f'Could not find a genes TSV with {nrows} rows for {sample_id} (needed for matrix rows)')
        genes_gz = picked
        print(f'[{sample_id}] Using genes file: {genes_gz.name}')

    # write prepared files
    gunzip_to(bc_gz, out_dir / f'{sample_id}_barcodes.tsv')
    gunzip_to(genes_gz, out_dir / f'{sample_id}_genes.tsv')
    gunzip_to(mx_gz, out_dir / f'{sample_id}_matrix.mtx')
    gunzip_to(meta_gz, out_dir / f'{sample_id}_meta.tsv')


def remote_content_length(url: str) -> int | None:
    try:
        req = Request(url, method='HEAD')
        with urlopen(req) as r:
            cl = r.headers.get('Content-Length')
        return int(cl) if cl else None
    except Exception as e:
        print(f"[Warn] Could not fetch Content-Length for {url}: {e}")
        return None


def download_if_needed(url: str, out_path: Path) -> None:
    expected = remote_content_length(url)

    if out_path.exists() and out_path.stat().st_size > 0:
        if expected is None or out_path.stat().st_size == expected:
            print(f'[Cache hit] {out_path}')
            return
        if expected is not None and out_path.stat().st_size < expected:
            print(f"[Partial] {out_path} is {out_path.stat().st_size} bytes, expected {expected}; re-downloading.")
            out_path.unlink()

    part_path = out_path.with_suffix(out_path.suffix + '.part')
    if part_path.exists():
        print(f'[Cleanup] Removing partial download {part_path}')
        part_path.unlink()

    print(f'Downloading GEO archive -> {out_path} ...')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(url) as r, open(part_path, 'wb') as f:
        # Stream the download to disk (avoid loading ~820MB into RAM).
        shutil.copyfileobj(r, f, length=1024 * 1024)

    os.replace(part_path, out_path)
    size = out_path.stat().st_size
    print(f'[Saved] {out_path} ({size/1e6:.1f} MB)')


def extract_if_needed(archive_path: Path, extract_dir: Path) -> None:
    if extract_dir.exists() and any(extract_dir.glob('GSM*')):
        print(f'[Cache hit] extracted files exist in {extract_dir}')
        return
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {archive_path} -> {extract_dir} ...')
    with tarfile.open(archive_path, 'r:*') as tf:
        tf.extractall(path=extract_dir)


# Prepare
if (M5K_DIR / 'GSM4905335_matrix.mtx').exists() and (LM0_DIR / 'GSM4905342_matrix.mtx').exists():
    print('[Skip] Prepared raw sample folders already exist.')
else:
    download_if_needed(GEO_URL, ARCHIVE_PATH)
    extract_if_needed(ARCHIVE_PATH, EXTRACT_DIR)

    # For this project we use M5k and LM0
    ensure_prepared_sample(EXTRACT_DIR, sample_id='GSM4905335', tag='5k', out_dir=M5K_DIR)
    ensure_prepared_sample(EXTRACT_DIR, sample_id='GSM4905342', tag='LM0', out_dir=LM0_DIR)

print('Prepared raw dirs:')
print(' -', M5K_DIR)
print(' -', LM0_DIR)


## 4. Imports and helper functions
Reusable utilities for plotting, IO, and QC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import io
import anndata as ad


# Plotting helper
def my_savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"[Saved] {path}")


# QC violin plots
def plot_qc_metrics(adata: sc.AnnData, metrics: list, groupby: str, save_path: Path) -> None:
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    if len(metrics) == 1:
        axes = [axes]

    for i, metric in enumerate(metrics):
        sc.pl.violin(
            adata,
            metric,
            groupby=groupby,
            jitter=0.4,
            ax=axes[i],
            show=False,
            stripplot=True,
            rotation=90,
        )
        axes[i].set_xlabel(groupby.replace('_', ' ').title())
        axes[i].set_title(metric.replace('_', ' ').title())

    plt.tight_layout()
    my_savefig(save_path)
    plt.show()


# Raw loader (GSE161363 style MTX)
def load_sample_gse161363(
    sample_dir: str,
    batch_name: str | None = None,
) -> ad.AnnData:
    sample_dir = Path(sample_dir)

    files = list(sample_dir.glob("GSM*_barcodes.tsv"))
    if len(files) != 1:
        raise ValueError(f"Cannot uniquely identify prefix in {sample_dir}")
    prefix = files[0].name.replace("_barcodes.tsv", "")

    barcodes_file = sample_dir / f"{prefix}_barcodes.tsv"
    genes_file = sample_dir / f"{prefix}_genes.tsv"
    matrix_file = sample_dir / f"{prefix}_matrix.mtx"
    meta_file = sample_dir / f"{prefix}_meta.tsv"

    mtx = io.mmread(matrix_file).T.tocsr()
    barcodes = pd.read_csv(barcodes_file, header=None)[0].tolist()
    genes = pd.read_csv(genes_file, sep="	", header=None)
    genes.columns = ["gene_id", "gene_symbol"]

    adata = sc.AnnData(X=mtx)
    adata.obs_names = barcodes
    adata.var_names = genes["gene_symbol"]
    adata.var["gene_id"] = genes["gene_id"].values
    adata.var_names_make_unique()

    if meta_file.exists():
        meta = pd.read_csv(meta_file, sep='	')
        index_col = meta.columns[0]
        meta = meta.set_index(index_col)
        adata.obs = adata.obs.join(meta, how='left')

    if batch_name is not None:
        adata.obs["batch"] = batch_name

    return adata


# Clean obs rows with missing metadata
def clean_adata_obs(
    adata: ad.AnnData,
    columns: list,
    how: str = "any",
) -> ad.AnnData:
    missing_cols = [c for c in columns if c not in adata.obs.columns]
    if missing_cols:
        raise KeyError(f"Missing columns in adata.obs: {missing_cols}")

    df_check = adata.obs[columns].copy()
    df_check.replace("", np.nan, inplace=True)

    if how == "all":
        mask_remove = df_check.isna().all(axis=1)
    elif how == "any":
        mask_remove = df_check.isna().any(axis=1)
    else:
        raise ValueError("how must be 'all' or 'any'")

    filtered = adata[~mask_remove, :].copy()
    for col in filtered.obs.select_dtypes(include='category').columns:
        filtered.obs[col] = filtered.obs[col].cat.remove_unused_categories()

    return filtered


## 5. Load data
Load M5k and LM0 raw inputs.

In [ ]:
adata_5k = load_sample_gse161363(DIR_RAW / "M5k", batch_name="M5k")
adata_lm0 = load_sample_gse161363(DIR_RAW / "LM0", batch_name="LM0")

print(adata_5k)
print(adata_lm0)


## 6. Metadata cleanup and fill
Create clean and filled versions of LM0; define LG_Present labels for M5k.

In [ ]:
# Fill LM0 missing metadata to match the original `Merged_fill` objects.
# GEO LM0 meta file contains limited columns; downstream expects the full set.

adata_lm0_fill = adata_lm0.copy()

# Ensure key numeric columns exist and are numeric
for col in ['LineageGroup', 'TreeMetRate']:
    if col not in adata_lm0_fill.obs.columns:
        adata_lm0_fill.obs[col] = np.nan
    adata_lm0_fill.obs[col] = pd.to_numeric(adata_lm0_fill.obs[col], errors='coerce')

adata_lm0_fill.obs['LineageGroup'] = adata_lm0_fill.obs['LineageGroup'].fillna(0.0)
adata_lm0_fill.obs['TreeMetRate'] = adata_lm0_fill.obs['TreeMetRate'].fillna(0.0)

# LM0 sample labels: derived from LG_Present (Engrafted / NotEngrafted)
if 'LG_Present' not in adata_lm0_fill.obs.columns:
    adata_lm0_fill.obs['LG_Present'] = 'NotEngrafted'

lm0_group = np.where(
    adata_lm0_fill.obs['LG_Present'].astype(str).eq('Engrafted'),
    'LM0_Engraft',
    'LM0_NotEngraft',
)

# These columns are present for M5k but absent for LM0 in GEO; fill with 0 as in the original pipeline.
for col in ['TS_UMI', 'NUM_INTBC', 'StaticMetScore', 'TissueDispersion', 'scTreeMetRate']:
    if col not in adata_lm0_fill.obs.columns:
        adata_lm0_fill.obs[col] = 0.0
    adata_lm0_fill.obs[col] = pd.to_numeric(adata_lm0_fill.obs[col], errors='coerce').fillna(0.0)

adata_lm0_fill.obs['sampleID'] = pd.Categorical(lm0_group)
adata_lm0_fill.obs['Sample2'] = pd.Categorical(lm0_group)

# Clean versions by removing missing lineage fields
adata_5k_clean = clean_adata_obs(adata_5k, columns=['LineageGroup', 'scTreeMetRate'], how='any')
adata_lm0_clean = clean_adata_obs(adata_lm0, columns=['LineageGroup', 'TreeMetRate'], how='any')

# M5k lineage label (as a categorical string)
lineage_group = adata_5k_clean.obs['LineageGroup']
is_present = pd.notna(lineage_group) & (lineage_group >= 1) & (lineage_group <= 100)
present_labels = lineage_group.fillna(0).astype(int).astype(str)
adata_5k_clean.obs['LG_Present'] = np.select([is_present], [present_labels], default='Unassigned')

print(adata_5k_clean)
print(adata_lm0_clean)
print(adata_lm0_fill)


## 7. Merge datasets
Merge clean M5k with filled LM0 for the integrated dataset.

In [ ]:
adata = ad.concat(
    [adata_5k_clean, adata_lm0_fill],
    join="outer",
    merge='first',
    label="dataset",
    keys=["M5k", "LM0"],
    index_unique=None
)

print(adata)


## 8. QC metrics and filtering
Compute QC metrics and apply filters.

In [ ]:
# Mito and ribo flags
adata.var['mt'] = adata.var_names.str.startswith(("MT-", "mt-"))
adata.var['ribo'] = adata.var_names.str.startswith(("RPS", "RPL"))

sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], inplace=True)

QC_METRICS = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo']
plot_qc_metrics(adata, QC_METRICS, groupby='dataset', save_path=DIR_FIG_SAVE / 'qc_violin.dataset.pdf')

# Cell filtering
cell_mask = (adata.obs['n_genes_by_counts'] >= MIN_GENES) &             (adata.obs['n_genes_by_counts'] <= MAX_GENES) &             (adata.obs['pct_counts_mt'] < MAX_MT_PCT)
adata = adata[cell_mask, :].copy()

# Gene filtering
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)

plot_qc_metrics(adata, QC_METRICS, groupby='dataset', save_path=DIR_FIG_SAVE / 'postqc_violin.dataset.pdf')

print(adata)


## 9. Clustering pipeline
Normalize, HVGs, PCA, Harmony, neighbors, UMAP, and Leiden.

In [ ]:
def run_cluster(
    adata: ad.AnnData,
    hvgs_batch_key: str | None = None,
    harmony_batch_key: str | None = None,
    n_pcs_neighbors: int = 40,
    leiden_res: list[float] | None = None,
) -> ad.AnnData:
    if leiden_res is None:
        leiden_res = [0.3, 0.6, 0.9, 1.2]

    adata.layers['raw_counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata.copy()

    sc.pp.highly_variable_genes(
        adata,
        min_mean=0.0125,
        max_mean=3,
        min_disp=0.5,
        batch_key=hvgs_batch_key,
    )

    sc.tl.pca(adata, svd_solver='arpack', n_comps=100, mask_var='highly_variable')

    if harmony_batch_key:
        sc.external.pp.harmony_integrate(
            adata,
            key=harmony_batch_key,
            basis='X_pca',
            adjusted_basis='X_pca_harmony',
            random_state=NEIGHBORS_UMAP_SEED,
            theta=HARMONY_THETA,
            max_iter_harmony=HARMONY_MAX_ITER,
        )
        use_rep = 'X_pca_harmony'
    else:
        use_rep = 'X_pca'

    sc.pp.neighbors(
        adata,
        n_neighbors=15,
        n_pcs=n_pcs_neighbors,
        use_rep=use_rep,
        random_state=NEIGHBORS_UMAP_SEED,
    )
    sc.tl.umap(adata, min_dist=0.5, spread=1.0, random_state=NEIGHBORS_UMAP_SEED)

    for res in leiden_res:
        sc.tl.leiden(
            adata,
            resolution=res,
            key_added=f'Leiden_Res_{res:.1f}',
            random_state=LEIDEN_SEED,
            flavor='igraph',
            n_iterations=2,
        )

    return adata


adata_clust = run_cluster(
    adata,
    hvgs_batch_key=None,
    harmony_batch_key='dataset',
    n_pcs_neighbors=N_PCS,
    leiden_res=LEIDEN_RES,
)

print(adata_clust)


## 10. Save outputs
Save the integrated final h5ad for downstream analysis.

In [ ]:
out_path = DIR_PROC_OUT / 'A549_integrated_final.h5ad'

if out_path.exists() and not OVERWRITE_OUTPUT:
    print(f'[Skip] {out_path} exists. Set OVERWRITE_OUTPUT=True to overwrite.')
else:
    # Enforce deterministic category ordering for reproducibility (matches legacy object)
    import pandas as pd

    SAMPLE2_ORDER = ['LL', 'Liv', 'M', 'RL', 'LM0_Engraft', 'LM0_NotEngraft']
    SAMPLEID_ORDER = ['LL', 'Liv', 'M1', 'M2', 'RE', 'RW', 'LM0_Engraft', 'LM0_NotEngraft']

    if 'Sample2' in adata_clust.obs.columns:
        v = adata_clust.obs['Sample2'].astype(str)
        adata_clust.obs['Sample2'] = pd.Categorical(v, categories=SAMPLE2_ORDER, ordered=False)
    if 'sampleID' in adata_clust.obs.columns:
        v = adata_clust.obs['sampleID'].astype(str)
        adata_clust.obs['sampleID'] = pd.Categorical(v, categories=SAMPLEID_ORDER, ordered=False)

    adata_clust.write(out_path)
    print(f'[Saved] {out_path}')


## 11. Reproducibility check (fingerprint)
This optional check computes a compact fingerprint of the output and compares it to the reference fingerprint shipped with the dissemination package.

Notes:
- Categorical labels (e.g., `Sample2`, `sampleID`, Leiden labels) are checked **exactly**.
- Continuous embeddings are checked with **numerically robust** summaries (UMAP can vary slightly across hardware/threads even with fixed seeds).


In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import anndata as ad
import numpy as np
from importlib.metadata import version


def _to_jsonable(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.ndarray,)):
        return x.tolist()
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    return x


def _md5_bytes(data: bytes) -> str:
    h = hashlib.md5()
    h.update(data)
    return h.hexdigest()


def _md5_array(a: np.ndarray) -> str:
    a = np.ascontiguousarray(a)
    return _md5_bytes(a.tobytes())


def _md5_series_values(s) -> dict:
    vals = [str(x) for x in s.tolist()]
    return {
        'values_md5': _md5_bytes(('\n'.join(vals)).encode('utf-8')),
        'n_unique': int(len(set(vals))),
    }


def _sample_md5_rounded(a: np.ndarray, decimals: int = 6, n_rows: int = 1024, n_cols: int = 32) -> str:
    a = np.asarray(a)
    if a.ndim != 2:
        a = a.reshape(a.shape[0], -1)
    r = min(n_rows, a.shape[0])
    c = min(n_cols, a.shape[1])
    r_idx = np.linspace(0, a.shape[0] - 1, num=r, dtype=int)
    c_idx = np.linspace(0, a.shape[1] - 1, num=c, dtype=int)
    sub = a[np.ix_(r_idx, c_idx)].astype(np.float64, copy=False)
    sub = np.round(sub, decimals=decimals)
    return _md5_array(sub)


def _umap_distance_quantiles(x: np.ndarray, seed: int = 0, sample_size: int = 2000, m: int = 256) -> list[float]:
    x = np.asarray(x)
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.choice(n, size=min(sample_size, n), replace=False)
    xs = x[idx].astype(np.float64)
    xs = xs - xs.mean(axis=0, keepdims=True)
    m = min(m, xs.shape[0])
    xs = xs[:m]
    d = xs[:, None, :] - xs[None, :, :]
    dist = np.sqrt((d * d).sum(axis=-1))
    tri = dist[np.triu_indices(m, 1)]
    q = np.quantile(tri, [0.05, 0.5, 0.95])
    return [float(v) for v in q]


def fingerprint_h5ad(path: Path) -> dict:
    a = ad.read_h5ad(path, backed='r')

    try:
        seed = int(NEIGHBORS_UMAP_SEED)
    except Exception:
        seed = 0

    fp = {
        'file': path.name,
        'shape': [int(a.n_obs), int(a.n_vars)],
        'versions': {
            'scanpy': version('scanpy'),
            'harmonypy': version('harmonypy'),
            'leidenalg': version('leidenalg'),
            'python-igraph': version('igraph'),
            'umap-learn': version('umap-learn'),
        },
        'obsm': {
            'X_pca_harmony_sample_md5_round6': _sample_md5_rounded(np.asarray(a.obsm['X_pca_harmony']), decimals=6),
            'X_umap_dist_quantiles': _umap_distance_quantiles(np.asarray(a.obsm['X_umap']), seed=seed),
        },
        'obs': {},
        'neighbors_params': dict(a.uns.get('neighbors', {}).get('params', {})),
    }

    for k in [
        'dataset',
        'LG_Present',
        'Leiden_Res_0.3',
        'Leiden_Res_0.6',
        'Leiden_Res_0.9',
        'Leiden_Res_1.2',
        'sampleID',
        'Sample2',
    ]:
        if k in a.obs.columns:
            fp['obs'][k] = _md5_series_values(a.obs[k].astype(str))

    a.file.close()
    return _to_jsonable(fp)


out_path = DIR_PROC_OUT / 'A549_integrated_final.h5ad'
run_fp = fingerprint_h5ad(out_path)

# Write run fingerprint for the record
fp_out = DIR_SAVE / 'fingerprint.A549_integrated_final.json'
fp_out.write_text(json.dumps(run_fp, indent=2, sort_keys=True) + '\n')
print('[Saved]', fp_out)

# Compare to reference fingerprint if present
ref_fp_path = DIR_DATA / 'reference' / 'reference_fingerprint.A549_integrated_final.json'
if ref_fp_path.exists():
    ref_fp = json.loads(ref_fp_path.read_text())

    ok = True
    for k in ['shape', 'versions', 'obs']:
        if run_fp.get(k) != ref_fp.get(k):
            ok = False
            print(f'[Mismatch] {k}')

    ref_obsm = ref_fp.get('obsm', {})
    run_obsm = run_fp.get('obsm', {})

    if run_obsm.get('X_pca_harmony_sample_md5_round6') != ref_obsm.get('X_pca_harmony_sample_md5_round6'):
        ok = False
        print('[Mismatch] obsm.X_pca_harmony_sample_md5_round6')

    UMAP_Q_ATOL = 0.05
    rq = np.asarray(run_obsm.get('X_umap_dist_quantiles', []), dtype=float)
    fq = np.asarray(ref_obsm.get('X_umap_dist_quantiles', []), dtype=float)
    if rq.shape != fq.shape or (rq.size and not np.allclose(rq, fq, rtol=0, atol=UMAP_Q_ATOL)):
        ok = False
        print('[Mismatch] obsm.X_umap_dist_quantiles')
        print('  run:', rq.tolist())
        print('  ref:', fq.tolist())
        print('  atol:', UMAP_Q_ATOL)

    if ok:
        print('[PASS] Fingerprint matches reference (labels exact; embeddings within tolerance).')
    else:
        print('[FAIL] Fingerprint mismatch. If this is unexpected, restart the kernel and run all cells (thread/env vars must be set before imports).')
        print('Reference:', ref_fp_path)
else:
    print('[Info] No reference fingerprint found at', ref_fp_path)
